# 🗣️ Sauti TTS — Swahili Text-to-Speech Demo

Fine-tune F5-TTS on Google WaxalNLP for high-quality Swahili speech synthesis.

**Pipeline:**
1. Download & preprocess WaxalNLP Swahili TTS data
2. Fine-tune F5-TTS v1 Base
3. Generate Swahili speech with voice cloning
4. Evaluate quality (MOS, CER, speaker similarity)

## Setup

In [ ]:
# Install dependencies (run once)
!pip install -q torch torchaudio transformers datasets accelerate
!pip install -q soundfile librosa pyloudnorm
!pip install -q peft wandb tensorboard
!pip install -q jiwer openai-whisper resemblyzer
!pip install -q pyyaml rich num2words

# Install F5-TTS from source
!git clone https://github.com/SWivid/F5-TTS.git
%cd F5-TTS
!pip install -e .
%cd ..

In [ ]:
import os
import sys
sys.path.insert(0, '..')

import torch
import torchaudio
import numpy as np
from IPython.display import Audio, display

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Step 1: Download & Prepare WaxalNLP Swahili Data

In [ ]:
from datasets import load_dataset

# Load Swahili TTS data from Google WaxalNLP
dataset = load_dataset("google/WaxalNLP", "swa_tts", trust_remote_code=True)
print(dataset)
print(f"\nTrain: {len(dataset['train'])} samples")
print(f"Validation: {len(dataset['validation'])} samples")
print(f"Test: {len(dataset['test'])} samples")

# Inspect a sample
sample = dataset['train'][0]
print(f"\nSample fields: {list(sample.keys())}")
print(f"Text: {sample['text']}")
print(f"Speaker: {sample['speaker_id']}")
print(f"Gender: {sample['gender']}")
print(f"Audio SR: {sample['audio']['sampling_rate']}")
print(f"Audio length: {len(sample['audio']['array']) / sample['audio']['sampling_rate']:.2f}s")

In [ ]:
# Listen to a sample
audio_array = np.array(sample['audio']['array'])
sr = sample['audio']['sampling_rate']
print(f"Playing: {sample['text']}")
display(Audio(audio_array, rate=sr))

In [ ]:
# Full data preparation pipeline
from sauti_tts.data import WaxalSwahiliDataset

data_pipeline = WaxalSwahiliDataset(
    output_dir="../data/waxal_swahili",
    sample_rate=24000,
    min_duration=1.0,
    max_duration=30.0,
    normalize=True,
    trim=True,
)

stats = data_pipeline.download_and_prepare()
print(f"\nTotal samples: {stats.total_samples}")
print(f"Total duration: {stats.total_duration_hours:.2f} hours")
print(f"Speakers: {stats.num_speakers}")

## Step 2: Fine-tune F5-TTS

In [ ]:
from sauti_tts.model import SautiTTS, SautiTTSConfig
from sauti_tts.utils import count_parameters

# Build model
config = SautiTTSConfig(
    model_type="F5TTS_v1_Base",
    pretrained_path=None,  # Will auto-download from HuggingFace
    use_lora=False,        # Set True for lower VRAM (12GB)
    lora_rank=16,
)

sauti = SautiTTS(config)
model = sauti.build_model()

params = count_parameters(model)
print(f"\nTotal parameters: {params['total']:,}")
print(f"Trainable: {params['trainable']:,} ({params['trainable_pct']:.1f}%)")

In [ ]:
# Option A: Train with F5-TTS built-in trainer (recommended)
from sauti_tts.trainer import SautiTrainer, TrainingConfig

training_config = TrainingConfig(
    exp_name="sauti_tts",
    output_dir="../ckpts/sauti_tts",
    dataset_dir="../data/waxal_swahili",
    
    # Training hyperparameters
    epochs=100,
    learning_rate=1e-5,
    batch_size_per_gpu=1600,  # Adjust for your GPU VRAM
    batch_size_type="frame",
    max_samples=64,
    
    # EMA (disable for short fine-tuning)
    use_ema=True,
    ema_decay=0.9999,
    
    # Checkpointing
    save_per_updates=5000,
    
    # Logging
    logger_type="tensorboard",
)

trainer = SautiTrainer(training_config, model)
trainer.train_with_f5tts()

In [ ]:
# Option B: Use F5-TTS CLI directly (simpler)
# Prepare dataset in F5-TTS format first:
!python F5-TTS/src/f5_tts/train/datasets/prepare_csv_wavs.py \
    ../data/waxal_swahili/metadata.csv \
    F5-TTS/data/sauti_tts

# Copy pretrained checkpoint
!mkdir -p ../ckpts/sauti_tts
# The model will be auto-downloaded on first use

# Train (adjust batch_size for your GPU)
!accelerate launch F5-TTS/src/f5_tts/train/train.py \
    --config-name F5TTS_v1_Base.yaml \
    ++datasets.name=sauti_tts \
    ++datasets.batch_size_per_gpu=1600 \
    ++optim.learning_rate=1e-5 \
    ++trainer.epochs=100

## Step 3: Generate Swahili Speech

In [ ]:
from scripts.inference import SautiInference

# Load fine-tuned model
engine = SautiInference(
    checkpoint_path="../ckpts/sauti_tts/model_last.pt",
)

# Test sentences in Swahili
test_texts = [
    "Habari za asubuhi. Karibu kwenye mfumo wetu wa Sauti.",
    "Kenya ni nchi nzuri yenye watu wengi wanaopenda amani.",
    "Teknolojia ya akili bandia inabadilisha dunia yetu.",
    "Watoto wanajifunza lugha kwa haraka sana.",
    "Mvua inanyesha na maua yanachanua kwa uzuri.",
]

# Use a reference audio from the dataset for voice cloning
ref_audio_path = "../data/waxal_swahili/wavs/swa_0.wav"  # Adjust path

for i, text in enumerate(test_texts):
    print(f"\n[{i+1}] {text}")
    output_path = f"../outputs/demo_{i}.wav"
    audio = engine.generate(
        text=text,
        ref_audio_path=ref_audio_path,
        output_path=output_path,
        nfe_steps=32,
    )
    display(Audio(audio, rate=24000))

## Step 4: Evaluate Quality

In [ ]:
from sauti_tts.metrics import SautiEvaluator

evaluator = SautiEvaluator(whisper_model="medium")

# Evaluate a single generated sample
gen_audio = np.random.randn(24000 * 3)  # Replace with actual generated audio
ref_audio = np.random.randn(24000 * 3)  # Replace with actual reference audio

result = evaluator.evaluate_sample(
    ref_audio=ref_audio,
    gen_audio=gen_audio,
    ref_text="Habari za asubuhi",
    sample_id="demo_0",
)

print(f"MOS: {result.mos_score:.2f}")
print(f"Speaker Similarity: {result.speaker_similarity:.3f}")
print(f"CER: {result.cer:.3f}")
print(f"WER: {result.wer:.3f}")

In [ ]:
# Full evaluation on test set
!python ../scripts/evaluate.py \
    --checkpoint ../ckpts/sauti_tts/model_last.pt \
    --test_set ../data/waxal_swahili/test_metadata.csv \
    --ref_audio ../data/waxal_swahili/wavs/swa_0.wav \
    --output_dir ../outputs/eval/ \
    --max_samples 50

## Swahili Text Normalization Demo

In [ ]:
from sauti_tts.utils import normalize_swahili_text, number_to_swahili

# Number conversion
for n in [0, 5, 15, 42, 100, 256, 1000, 2026, 1000000]:
    print(f"{n:>10,} → {number_to_swahili(n)}")

# Text normalization
test_texts = [
    "Dkt. Amina ana Ksh 5,000 na USD 100.",
    "Bw. Juma alisafiri km 250 hadi Nairobi.",
    "Tarehe 15 mwaka 2026, Prof. Mwangi alitoa hotuba.",
]

for text in test_texts:
    normalized = normalize_swahili_text(text)
    print(f"\nOriginal:   {text}")
    print(f"Normalized: {normalized}")